[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/data-engineering-from-scratch-to-advanced/blob/main/notebooks/topics/12_spark-internals.ipynb)

*Data Engineering from Scratch to Advanced · Notebook 12 of 21*

# Spark Tuning & Internals

Make Spark fast and stop it crashing: join strategies, broadcast, skew and salting, shuffle tuning, adaptive execution, memory, out-of-memory errors, file sizes, reading the Spark UI, and real debugging.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** When a job crawls, one task is usually carrying everyone's load. Find it before you buy more machines.

**What interviewers really test here:** Join strategies, skew and salting, shuffle tuning, out-of-memory errors and how you debug with the Spark UI.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Join strategies: the biggest performance lever *(Core)*
- [ ] 2. Skew: why jobs get stuck at 99% *(Core)*
- [ ] 3. Shuffle tuning and adaptive execution *(Core)*
- [ ] 4. Memory and out-of-memory errors *(Advanced)*
- [ ] 5. A debugging routine for any slow job *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Settings worth knowing

| Setting | Controls |
| --- | --- |
| `spark.sql.shuffle.partitions` | partitions after a shuffle (default 200) |
| `spark.sql.adaptive.enabled` | Adaptive Query Execution |
| `spark.sql.autoBroadcastJoinThreshold` | the largest table auto-broadcast |
| `spark.sql.files.maxPartitionBytes` | input partition size |
| `spark.executor.memoryOverhead` | non-heap memory |
| `spark.sql.sources.partitionOverwriteMode` | `dynamic` overwrites only the partitions written |

---
# Part 2: Lessons, step by step

## Lesson 1: Join strategies: the biggest performance lever

*Level: Core*

Joins usually dominate a Spark job's cost. Spark chooses among these:



| Join | When | Shuffle? |
| --- | --- | --- |
| **Broadcast hash** | one side's small (default limit 10 MB) | none |
| **Sort-merge** | both large (the default) | both sides |
| **Shuffle hash** | one side moderately small | both sides |
| **Bucketed** | both bucketed the same way | none |



**Broadcast** sends the small table to every executor, so each partition of the big table joins locally:



```python
from pyspark.sql import functions as F
big.join(F.broadcast(small), "key")
```



In the lab you can see `BroadcastHashJoin` in the plan. Don't broadcast something that's too big, though, as it has to fit in the driver and every executor.



**Bucketing pays once.** Joining 100 GB with 900 GB daily for 30 days shuffles 30,000 GB. Bucketed, you shuffle 1,000 GB once, at write time.



**Filter via a join: dynamic partition pruning.** Joining a fact table to a dimension filtered to one day lets Spark read only that day's partitions: 1 instead of 5.



**Your checklist for a slow join:**



1. Can you filter or select columns first?
2. Can one side be broadcast?
3. Is the key skewed? Are there NULLs?
4. Is there a many-to-many explosion on the key?

**✏️ Your turn:** you now have what you need for [Q5](#q5), [Q6](#q6), [Q15](#q15), [Q16](#q16). Try one now, or carry on and do them in Part 3.

## Lesson 2: Skew: why jobs get stuck at 99%

*Level: Core*

199 tasks finish in a minute, and one runs for two hours. That's **data skew**, the most common real Spark problem.



**Spot it in the Spark UI:** open the stage, and compare the median and max under Input, Shuffle Read and Duration. A max far above the median means skew. With tasks (60 s, 900 MB) when the median is (11 s, 100 MB), it's skew. A slow task with normal input (50 s, 100 MB) points to a sick node instead.



**Causes:** a hot key (a big customer), or NULLs, which all hash to one partition. If 60% of keys are NULL, filter those first; they can't match in an inner join anyway.



| Fix | How | Trade-off |
| --- | --- | --- |
| Filter NULLs | simply drop or treat separately | trivial; try first |
| **AQE skew join** | Spark splits big partitions automatically | the easy option in Spark 3+ |
| Broadcast | no shuffle, so no skew | needs a small side |
| Isolate hot keys | handle them separately, union the results | more code |
| **Salting** | add a random suffix to split a key | replicates the other side |



**Salting by hand.** Key "us" has 3 rows, and with 2 salts it becomes us#0 and us#1, with the largest group going from 3 to 2. The other table must have every salted version of each key: "us" appears as both us#0 and us#1, so the join still matches. In the lab, salting 10 ways reduces the largest group of a hot key from 9,500 to 1,000.



```python
salted = big.withColumn("k", F.concat("key", F.lit("#"), (F.rand() * 10).cast("int")))
small2 = small.crossJoin(spark.range(10)).withColumn("k", F.concat("key", F.lit("#"), "id"))
salted.join(small2, "k")
```



**AQE decides skew** with a rule: bigger than 5 times the median, and bigger than 256 MB (the defaults).

**✏️ Your turn:** you now have what you need for [Q7](#q7), [Q1](#q1), [Q8](#q8), [Q19](#q19), [Q2](#q2). Try one now, or carry on and do them in Part 3.

## Lesson 3: Shuffle tuning and adaptive execution

*Level: Core*

`spark.sql.shuffle.partitions` sets the partitions after a shuffle, and it defaults to **200**, which suits almost nobody:



| Shuffle size | Average with 200 | Problem | At 128 MB each |
| --- | --- | --- | --- |
| 1 TB | 5,000 MB | huge tasks, spilling | 7,813 partitions |
| 2 GB | 10 MB | tiny tasks and files | 16 partitions |



**Adaptive Query Execution (AQE)** re-optimises mid-job, using real sizes. It's been on by default since Spark 3.2:



- **Coalesces small partitions:** 10, 20, 30, 100, 5, 5 with a 64 MB target becomes 60, 100, 10.
- **Splits skewed partitions.**
- **Switches join strategy,** for example to broadcast if a side turned out small after filtering.



So set a high shuffle partition count, and let AQE merge down. That usually beats guessing.



**Spilling:** when sorting data doesn't fit in memory, Spark writes sorted runs to disk and merges them. 1,000 MB with 300 MB of memory makes 4 runs. It still works, but it's slow, so look for "Spill" in the UI.



**Writing files.** With `partitionBy`, each task writes one file per value it holds: 200 tasks × 30 days = **6,000 small files**. Fix it with repartition first:



```python
df.repartition("day").write.partitionBy("day").parquet(path)   # 30 files, one per day
```



If a day's too large for one file, add a salt, or set `maxRecordsPerFile`.



A window with no `partitionBy` moves everything to one task: Spark warns you about it, and it's a performance trap.

**✏️ Your turn:** you now have what you need for [Q9](#q9), [Q10](#q10), [Q17](#q17), [Q13](#q13), [Q4](#q4), [Q18](#q18). Try one now, or carry on and do them in Part 3.

## Lesson 4: Memory and out-of-memory errors

*Level: Advanced*

An 8 GB executor's memory is divided like this:



```text
8 GB
├─ 300 MB reserved
└─ the rest (7.71 GB)
   ├─ 60% unified: 4.62 GB
   │   ├─ execution (joins, sorts, shuffles)
   │   └─ storage (cache); they borrow from each other
   └─ 40% user memory: 3.08 GB (your objects, UDFs)
```



On top of this, there's **overhead memory** outside the JVM, including Python workers. Containers killed by YARN or Kubernetes for exceeding limits usually need more overhead, not heap.



| Error | Likely cause | Try this |
| --- | --- | --- |
| Driver OOM | `collect`, `toPandas`, big broadcast | write out instead; lower broadcast threshold |
| Executor OOM on one task | skew | fix the skew |
| Executor OOM on many tasks | partitions too big | more partitions |
| Container killed | overhead exceeded | raise `memoryOverhead` |
| Shuffle fetch failed | executor died or was overloaded | look at why it died; often memory |



**Never treat more memory as the first answer.** It's expensive, and it hides the cause.



**Dynamic allocation** scales executors with the backlog: 100 pending tasks at 4 cores wants 25, capped at a limit.



**Cost.** 10 nodes for 2 hours costs the same as 20 nodes for 1 hour, if scaling's perfect, and it finishes twice as fast. If doubling nodes only cuts the time to 1.5 hours, it costs 50% more.

**✏️ Your turn:** you now have what you need for [Q11](#q11), [Q12](#q12), [Q14](#q14), [Q3](#q3). Try one now, or carry on and do them in Part 3.

## Lesson 5: A debugging routine for any slow job

*Level: Advanced*

**Follow the same steps every time.**



1. **Find the slowest stage** in the Spark UI. If shuffle takes 80 of 100 seconds, even making it infinitely fast gives a limit of 5×. That's where you start, not on the 10-second read.
2. **Look at task distribution:** median against max for duration, input and shuffle read.
3. **Read the SQL tab's plan:** rows at each step. A join outputting far more rows than its inputs is an explosion.
4. **Check for spills, and garbage collection time.** GC above about 10% of task time means memory pressure.



| Symptom | Diagnosis |
| --- | --- |
| One task much slower, more input | skew |
| One task slower, same input | bad node, speculation helps |
| All tasks slow, high GC | too little memory, partitions too big |
| Thousands of tiny tasks | too many partitions, or small files |
| Output rows far above input | many-to-many join |
| Long time reading | missing pruning, small files, a CSV source |



**The order to try fixes:** read less data (filters, columns, partitions), then fix the join, then fix skew, then tune settings, and only then add hardware.

**✏️ Your turn:** you now have what you need for [Q20](#q20). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Null Keys Cause Skew

🟢 Warm-up · *The classic surprise*

Return the share of rows with a NULL join key, rounded to 4 decimals. Those rows hash to one partition and never match in an inner join, so filter them first.

**Examples**

- `keys = [1, None, None, 2, None]` → `0.6`

In [ ]:
def null_share(keys):
    """
    Return the share of rows with a NULL join key, rounded to 4 decimals. Those
    rows hash to one partition and never match in an inner join, so filter them
    first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(null_share, [
    (([1, None, None, 2, None],), 0.6),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`count`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop.

```python
def null_share(keys):
    nulls = 0
    for k in keys:
        if k is None:
            nulls += 1
    return round(nulls / len(keys), 4)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `count`.

```python
def null_share(keys):
    return round(keys.count(None) / len(keys), 4)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(n) time and O(1) space."

"I can do better: `count`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/null-keys-cause-skew)

<a id="q2"></a>
### Q2. Spot the Straggler Task

🟢 Warm-up · *Read the Spark UI*

Tasks have (duration, input MB). Return indexes of tasks slower than 2 × median duration, labelled 'skew' if input is also over 2 × median input, else 'slow node'.

**Examples**

- `tasks = [(10, 100), (11, 110), (9, 90), (60, 900), (50, 100)]` → `[[3, 'skew'], [4, 'slow node']]`

In [ ]:
def stragglers(tasks):
    """
    Tasks have (duration, input MB). Return indexes of tasks slower than 2 ×
    median duration, labelled 'skew' if input is also over 2 × median input,
    else 'slow node'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(stragglers, [
    (([(10, 100), (11, 110), (9, 90), (60, 900), (50, 100)],), [[3, 'skew'], [4, 'slow node']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort for medians, then a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`statistics.median`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Sort for medians, then a loop.

```python
def stragglers(tasks):
    d = sorted(t[0] for t in tasks)
    i = sorted(t[1] for t in tasks)
    med_d, med_i = d[len(d) // 2], i[len(i) // 2]
    out = []
    for n, (dur, mb) in enumerate(tasks):
        if dur > 2 * med_d:
            out.append([n, "skew" if mb > 2 * med_i else "slow node"])
    return out
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `statistics.median`.

```python
def stragglers(tasks):
    import statistics
    md, mi = statistics.median(t[0] for t in tasks), statistics.median(t[1] for t in tasks)
    return [[n, "skew" if mb > 2 * mi else "slow node"] for n, (dur, mb) in enumerate(tasks) if dur > 2 * md]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort for medians, then a loop. That costs O(n log n) time and O(n) space."

"I can do better: `statistics.median`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/spot-the-straggler-task)

<a id="q3"></a>
### Q3. Cost of a Cluster Run

🟢 Warm-up · *Time × machines*

Return the cost of nodes × hours × a made-up hourly price, rounded to 2 decimals, for two options, and the cheaper's name. More nodes finishing faster can cost the same.

**Examples**

- `n1 = 10, h1 = 2, p1 = 0.5, n2 = 20, h2 = 1, p2 = 0.5` → `(10.0, 10.0, 'first')`
- `n1 = 10, h1 = 2, p1 = 0.5, n2 = 20, h2 = 1.5, p2 = 0.5` → `(10.0, 15.0, 'first')`

In [ ]:
def cluster_cost(n1, h1, p1, n2, h2, p2):
    """
    Return the cost of nodes × hours × a made-up hourly price, rounded to 2
    decimals, for two options, and the cheaper's name. More nodes finishing
    faster can cost the same.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cluster_cost, [
    ((10, 2, 0.5, 20, 1, 0.5), (10.0, 10.0, 'first')),
    ((10, 2, 0.5, 20, 1.5, 0.5), (10.0, 15.0, 'first')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute both, then compare.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Compute both, then compare.

```python
def cluster_cost(n1, h1, p1, n2, h2, p2):
    a = n1 * h1 * p1
    b = n2 * h2 * p2
    if a <= b:
        name = "first"
    else:
        name = "second"
    return (round(a, 2), round(b, 2), name)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def cluster_cost(n1, h1, p1, n2, h2, p2):
    a, b = n1 * h1 * p1, n2 * h2 * p2
    return (round(a, 2), round(b, 2), "first" if a <= b else "second")
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute both, then compare. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/cost-of-a-cluster-run)

<a id="q4"></a>
### Q4. Parquet Target File Count

🟢 Warm-up · *repartition before write*

Return repartition(n) for writing data\_gb at target\_mb per file, assuming compression shrinks by ratio. GB = 1000 MB.

**Examples**

- `data_gb = 100, target_mb = 256, ratio = 4` → `98`

In [ ]:
def write_partitions(data_gb, target_mb, ratio):
    """
    Return repartition(n) for writing data_gb at target_mb per file, assuming
    compression shrinks by ratio. GB = 1000 MB.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(write_partitions, [
    ((100, 256, 4), 98),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Step by step.

```python
def write_partitions(data_gb, target_mb, ratio):
    on_disk = data_gb * 1000 / ratio
    return math.ceil(on_disk / target_mb)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def write_partitions(data_gb, target_mb, ratio):
    return math.ceil(data_gb * 1000 / ratio / target_mb)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step by step. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/parquet-target-file-count)

<a id="q5"></a>
### Q5. Pick a Join Strategy

🟡 Interview standard · *What Spark chooses*

Return 'broadcast' if the smaller side (MB) is at or below threshold (default 10 MB in Spark), 'bucketed' if both are bucketed the same, otherwise 'sort-merge'.

**Examples**

- `a_mb = 5, b_mb = 1000000, threshold = 10, bucketed = False` → `'broadcast'`
- `a_mb = 500, b_mb = 1000000, threshold = 10, bucketed = True` → `'bucketed'`
- `a_mb = 500, b_mb = 1000000, threshold = 10, bucketed = False` → `'sort-merge'`

In [ ]:
def spark_join(a_mb, b_mb, threshold, bucketed):
    """
    Return 'broadcast' if the smaller side (MB) is at or below threshold
    (default 10 MB in Spark), 'bucketed' if both are bucketed the same,
    otherwise 'sort-merge'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(spark_join, [
    ((5, 1000000, 10, False), 'broadcast'),
    ((500, 1000000, 10, True), 'bucketed'),
    ((500, 1000000, 10, False), 'sort-merge'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Ifs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A conditional expression.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Ifs.

```python
def spark_join(a_mb, b_mb, threshold, bucketed):
    if min(a_mb, b_mb) <= threshold:
        return "broadcast"
    if bucketed:
        return "bucketed"
    return "sort-merge"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A conditional expression.

```python
def spark_join(a_mb, b_mb, threshold, bucketed):
    return "broadcast" if min(a_mb, b_mb) <= threshold else "bucketed" if bucketed else "sort-merge"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Ifs. That costs O(1) time and O(1) space."

"I can do better: A conditional expression. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/pick-a-join-strategy)

<a id="q6"></a>
### Q6. Broadcast Join Simulation

🟡 Interview standard · *Each partition joins locally*

Each partition of the big table joins a small dictionary, with no shuffle. Return the joined rows per partition, dropping unmatched.

**Examples**

- `parts = [[(1, 'x'), (2, 'y')], [(1, 'z'), (9, 'q')]], small = {1: 'A', 2: 'B'}` → `[[[1, 'x', 'A'], [2, 'y', 'B']], [[1, 'z', 'A']]]`

In [ ]:
def broadcast_join(parts, small):
    """
    Each partition of the big table joins a small dictionary, with no shuffle.
    Return the joined rows per partition, dropping unmatched.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(broadcast_join, [
    (([[(1, 'x'), (2, 'y')], [(1, 'z'), (9, 'q')]], {1: 'A', 2: 'B'}), [[[1, 'x', 'A'], [2, 'y', 'B']], [[1, 'z', 'A']]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loops scanning the small table.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary lookup for each row.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Nested loops scanning the small table.

```python
def broadcast_join(parts, small):
    out = []
    for p in parts:
        joined = []
        for k, v in p:
            for sk in small:
                if sk == k:
                    joined.append([k, v, small[sk]])
        out.append(joined)
    return out
```

**Cost:** O(k·s) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary lookup for each row.

```python
def broadcast_join(parts, small):
    return [[[k, v, small[k]] for k, v in p if k in small] for p in parts]
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loops scanning the small table. That costs O(k·s) time and O(k) space."

"I can do better: A dictionary lookup for each row. That's O(k) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/broadcast-join-simulation)

<a id="q7"></a>
### Q7. Find the Skewed Partitions

🟡 Interview standard · *Spark's AQE rule*

A partition is skewed if it's larger than factor × median and larger than min\_mb. Return the skewed indexes.

**Examples**

- `sizes = [100, 120, 110, 5000, 90], factor = 5, min_mb = 256` → `[3]`
- `sizes = [10, 10, 100], factor = 5, min_mb = 256` → `[]`

In [ ]:
def skewed(sizes, factor, min_mb):
    """
    A partition is skewed if it's larger than factor × median and larger than
    min_mb. Return the skewed indexes.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(skewed, [
    (([100, 120, 110, 5000, 90], 5, 256), [3]),
    (([10, 10, 100], 5, 256), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort for the median, then a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`statistics.median`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Sort for the median, then a loop.

```python
def skewed(sizes, factor, min_mb):
    s = sorted(sizes)
    n = len(s)
    median = s[n // 2] if n % 2 else (s[n // 2 - 1] + s[n // 2]) / 2
    return [i for i, v in enumerate(sizes) if v > factor * median and v > min_mb]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `statistics.median`.

```python
def skewed(sizes, factor, min_mb):
    import statistics
    m = statistics.median(sizes)
    return [i for i, v in enumerate(sizes) if v > max(factor * m, min_mb)]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort for the median, then a loop. That costs O(n log n) time and O(n) space."

"I can do better: `statistics.median`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/find-the-skewed-partitions)

<a id="q8"></a>
### Q8. Isolate the Hot Key

🟡 Interview standard · *Another fix for skew*

Keys with over limit rows are 'hot': broadcast-join them, and shuffle-join the rest. Return (hot keys sorted, rows in the normal path).

**Examples**

- `rows = [('a', 1), ('a', 1), ('a', 1), ('a', 1), ('a', 1), ('b', 1), ('c', 1), ('c', 1)], limit = 3` → `(['a'], 3)`

In [ ]:
def hot_keys(rows, limit):
    """
    Keys with over limit rows are 'hot': broadcast-join them, and shuffle-join
    the rest. Return (hot keys sorted, rows in the normal path).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(hot_keys, [
    (([('a', 1), ('a', 1), ('a', 1), ('a', 1), ('a', 1), ('b', 1), ('c', 1), ('c', 1)], 3), (['a'], 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count in a dictionary.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`Counter`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Count in a dictionary.

```python
def hot_keys(rows, limit):
    counts = {}
    for k, _ in rows:
        counts[k] = counts.get(k, 0) + 1
    hot = sorted(k for k, c in counts.items() if c > limit)
    normal = 0
    for k, _ in rows:
        if k not in hot:
            normal += 1
    return (hot, normal)
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `Counter`.

```python
def hot_keys(rows, limit):
    from collections import Counter
    c = Counter(k for k, _ in rows)
    hot = sorted(k for k, n in c.items() if n > limit)
    return (hot, sum(n for k, n in c.items() if k not in hot))
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count in a dictionary. That costs O(n) time and O(k) space."

"I can do better: `Counter`. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/isolate-the-hot-key)

<a id="q9"></a>
### Q9. Shuffle Partitions Setting

🟡 Interview standard · *The default is rarely right*

Target partition\_mb after a shuffle. Return the partition count (rounded up) for shuffle\_gb, and the average MB with the default of 200 partitions. GB = 1000 MB.

**Examples**

- `shuffle_gb = 1000, partition_mb = 128` → `(7813, 5000.0)`
- `shuffle_gb = 2, partition_mb = 128` → `(16, 10.0)`

In [ ]:
def shuffle_setting(shuffle_gb, partition_mb):
    """
    Target partition_mb after a shuffle. Return the partition count (rounded up)
    for shuffle_gb, and the average MB with the default of 200 partitions. GB =
    1000 MB.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(shuffle_setting, [
    ((1000, 128), (7813, 5000.0)),
    ((2, 128), (16, 10.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Step by step.

```python
def shuffle_setting(shuffle_gb, partition_mb):
    mb = shuffle_gb * 1000
    needed = math.ceil(mb / partition_mb)
    with_default = mb / 200
    return (needed, with_default)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def shuffle_setting(shuffle_gb, partition_mb):
    return (math.ceil(shuffle_gb * 1000 / partition_mb), shuffle_gb * 5.0)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step by step. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/shuffle-partitions-setting)

<a id="q10"></a>
### Q10. AQE: Coalesce Small Partitions

🟡 Interview standard · *Adaptive execution*

Adaptive Query Execution merges neighbouring small shuffle partitions up to target MB. Return the merged sizes.

**Examples**

- `sizes = [10, 20, 30, 100, 5, 5], target = 64` → `[60, 100, 10]`

In [ ]:
def aqe_coalesce(sizes, target):
    """
    Adaptive Query Execution merges neighbouring small shuffle partitions up to
    target MB. Return the merged sizes.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(aqe_coalesce, [
    (([10, 20, 30, 100, 5, 5], 64), [60, 100, 10]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A running total.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same algorithm, compactly.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A running total.

```python
def aqe_coalesce(sizes, target):
    out, current = [], 0
    for s in sizes:
        if current and current + s > target:
            out.append(current)
            current = 0
        current += s
    if current:
        out.append(current)
    return out
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same algorithm, compactly.

```python
def aqe_coalesce(sizes, target):
    out = []
    for s in sizes:
        if out and out[-1] + s <= target:
            out[-1] += s
        else:
            out.append(s)
    return out
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A running total. That costs O(n) time and O(m) space."

"I can do better: Same algorithm, compactly. That's O(n) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/aqe-coalesce-small-partitions)

<a id="q11"></a>
### Q11. Executor Memory Breakdown

🟡 Interview standard · *Where does memory go?*

Given executor GB: 300 MB reserved; of the rest, a fraction 0.6 is unified (execution + storage), half of which is initially storage. Return (unified GB, storage GB, user GB), rounded to 2 decimals. GB = 1024 MB.

**Examples**

- `gb = 8` → `(4.62, 2.31, 3.08)`

In [ ]:
def memory_split(gb):
    """
    Given executor GB: 300 MB reserved; of the rest, a fraction 0.6 is unified
    (execution + storage), half of which is initially storage. Return (unified
    GB, storage GB, user GB), rounded to 2 decimals. GB = 1024 MB.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(memory_split, [
    ((8,), (4.62, 2.31, 3.08)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Step by step.

```python
def memory_split(gb):
    usable = gb - 300 / 1024
    unified = usable * 0.6
    storage = unified * 0.5
    user = usable - unified
    return (round(unified, 2), round(storage, 2), round(user, 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def memory_split(gb):
    u = gb - 300 / 1024
    return (round(u * 0.6, 2), round(u * 0.3, 2), round(u * 0.4, 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step by step. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/executor-memory-breakdown)

<a id="q12"></a>
### Q12. Diagnose an Out-of-Memory

🟡 Interview standard · *Driver or executor?*

Given symptoms, return the likely fix. 'collect' → 'avoid collect; write out instead'. 'broadcast' → 'lower the broadcast threshold'. 'skew' → 'fix skew with salting or AQE'. 'many\_small\_tasks' → 'fewer partitions'. Otherwise 'increase memory only after finding the cause'.

**Examples**

- `symptom = 'collect'` → `'avoid collect; write out instead'`
- `symptom = 'skew'` → `'fix skew with salting or AQE'`
- `symptom = 'mystery'` → `'increase memory only after finding the cause'`

In [ ]:
def oom_fix(symptom):
    """
    Given symptoms, return the likely fix. 'collect' → 'avoid collect; write out
    instead'. 'broadcast' → 'lower the broadcast threshold'. 'skew' → 'fix skew
    with salting or AQE'. 'many_small_tasks' → 'fewer partitions'. Otherwise
    'increase memory only after finding the cause'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(oom_fix, [
    (('collect',), 'avoid collect; write out instead'),
    (('skew',), 'fix skew with salting or AQE'),
    (('mystery',), 'increase memory only after finding the cause'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Ifs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary with a default.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Ifs.

```python
def oom_fix(symptom):
    if symptom == "collect":
        return "avoid collect; write out instead"
    if symptom == "broadcast":
        return "lower the broadcast threshold"
    if symptom == "skew":
        return "fix skew with salting or AQE"
    if symptom == "many_small_tasks":
        return "fewer partitions"
    return "increase memory only after finding the cause"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary with a default.

```python
def oom_fix(symptom):
    return {"collect": "avoid collect; write out instead", "broadcast": "lower the broadcast threshold",
            "skew": "fix skew with salting or AQE", "many_small_tasks": "fewer partitions"}.get(symptom, "increase memory only after finding the cause")
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Ifs. That costs O(1) time and O(1) space."

"I can do better: A dictionary with a default. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/diagnose-an-out-of-memory)

<a id="q13"></a>
### Q13. Files Written per Partition

🟡 Interview standard · *How jobs make small files*

Writing with partitionBy: each task writes one file per partition value it holds. With t tasks and v values spread across all tasks, return the file count, and with repartition by that column first (one task per value).

**Examples**

- `t = 200, v = 30` → `(6000, 30)`

In [ ]:
def files_written(t, v):
    """
    Writing with partitionBy: each task writes one file per partition value it
    holds. With t tasks and v values spread across all tasks, return the file
    count, and with repartition by that column first (one task per value).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(files_written, [
    ((200, 30), (6000, 30)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Multiply.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Multiply.

```python
def files_written(t, v):
    naive = t * v
    fixed = v
    return (naive, fixed)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def files_written(t, v):
    return (t * v, v)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Multiply. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/files-written-per-partition)

<a id="q14"></a>
### Q14. Dynamic Allocation

🟡 Interview standard · *Scale to the backlog*

Pending tasks and cores per executor give the executors wanted, rounded up and capped by max. Remove idle ones down to min. Return the target.

**Examples**

- `pending = 100, cores = 4, min_e = 2, max_e = 10` → `10`
- `pending = 0, cores = 4, min_e = 2, max_e = 10` → `2`
- `pending = 20, cores = 4, min_e = 2, max_e = 10` → `5`

In [ ]:
def target_executors(pending, cores, min_e, max_e):
    """
    Pending tasks and cores per executor give the executors wanted, rounded up
    and capped by max. Remove idle ones down to min. Return the target.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(target_executors, [
    ((100, 4, 2, 10), 10),
    ((0, 4, 2, 10), 2),
    ((20, 4, 2, 10), 5),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Ifs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`min` and `max`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Ifs.

```python
def target_executors(pending, cores, min_e, max_e):
    want = pending // cores
    if pending % cores:
        want += 1
    if want > max_e:
        want = max_e
    if want < min_e:
        want = min_e
    return want
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `min` and `max`.

```python
def target_executors(pending, cores, min_e, max_e):
    return max(min_e, min(max_e, math.ceil(pending / cores)))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Ifs. That costs O(1) time and O(1) space."

"I can do better: `min` and `max`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/dynamic-allocation)

<a id="q15"></a>
### Q15. Bucketed Join Avoids Shuffle

🟡 Interview standard · *Pay once at write time*

Two tables bucketed the same way need no shuffle at join. Return shuffle GB for r runs without buckets (both tables each run), and with buckets (only once for writing).

**Examples**

- `a = 100, b = 900, runs = 30` → `(30000, 1000)`

In [ ]:
def bucket_savings(a, b, runs):
    """
    Two tables bucketed the same way need no shuffle at join. Return shuffle GB
    for r runs without buckets (both tables each run), and with buckets (only
    once for writing).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(bucket_savings, [
    ((100, 900, 30), (30000, 1000)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute each.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Compute each.

```python
def bucket_savings(a, b, runs):
    without = (a + b) * runs
    with_buckets = a + b
    return (without, with_buckets)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def bucket_savings(a, b, runs):
    return ((a + b) * runs, a + b)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute each. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/bucketed-join-avoids-shuffle)

<a id="q16"></a>
### Q16. Dynamic Partition Pruning

🟡 Interview standard · *Filter via the join*

A fact table is partitioned by day. A dimension filter keeps days. Return (partitions read without DPP, partitions read with DPP, sorted).

**Examples**

- `fact_days = [1, 2, 3, 4, 5], dim = {1: 'wk', 2: 'wk', 3: 'wk', 4: 'wk', 5: 'we'}, label = 'we'` → `(5, [5])`

In [ ]:
def dpp(fact_days, dim, label):
    """
    A fact table is partitioned by day. A dimension filter keeps days. Return
    (partitions read without DPP, partitions read with DPP, sorted).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(dpp, [
    (([1, 2, 3, 4, 5], {1: 'wk', 2: 'wk', 3: 'wk', 4: 'wk', 5: 'we'}, 'we'), (5, [5])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop.

```python
def dpp(fact_days, dim, label):
    keep = []
    for d in fact_days:
        if dim.get(d) == label:
            keep.append(d)
    return (len(fact_days), sorted(keep))
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def dpp(fact_days, dim, label):
    return (len(fact_days), sorted(d for d in fact_days if dim.get(d) == label))
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(n) time and O(k) space."

"I can do better: A comprehension. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/dynamic-partition-pruning)

<a id="q17"></a>
### Q17. Sort-Merge Spill

🟡 Interview standard · *Memory runs out, disk takes over*

Sorting data\_mb in memory\_mb requires spilling if it doesn't fit. Return (number of sorted runs on disk, whether it spilled).

**Examples**

- `data_mb = 1000, memory_mb = 300` → `(4, True)`
- `data_mb = 100, memory_mb = 300` → `(1, False)`

In [ ]:
def spill(data_mb, memory_mb):
    """
    Sorting data_mb in memory_mb requires spilling if it doesn't fit. Return
    (number of sorted runs on disk, whether it spilled).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(spill, [
    ((1000, 300), (4, True)),
    ((100, 300), (1, False)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Divide with a remainder check.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`ceil`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Divide with a remainder check.

```python
def spill(data_mb, memory_mb):
    if data_mb <= memory_mb:
        return (1, False)
    runs = data_mb // memory_mb
    if data_mb % memory_mb:
        runs += 1
    return (runs, True)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `ceil`.

```python
def spill(data_mb, memory_mb):
    return (math.ceil(data_mb / memory_mb), data_mb > memory_mb)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Divide with a remainder check. That costs O(1) time and O(1) space."

"I can do better: `ceil`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/sort-merge-spill)

<a id="q18"></a>
### Q18. Window With No Partition

🟡 Interview standard · *A hidden single-task trap*

A window with no PARTITION BY puts all rows on one task. Return (max rows on a task with partitioning by key, and without). Rows are keys.

**Examples**

- `keys = ['a', 'a', 'b', 'c']` → `(2, 4)`

In [ ]:
def window_task(keys):
    """
    A window with no PARTITION BY puts all rows on one task. Return (max rows on
    a task with partitioning by key, and without). Rows are keys.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(window_task, [
    ((['a', 'a', 'b', 'c'],), (2, 4)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count in a dictionary.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`Counter`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Count in a dictionary.

```python
def window_task(keys):
    counts = {}
    for k in keys:
        counts[k] = counts.get(k, 0) + 1
    return (max(counts.values()), len(keys))
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `Counter`.

```python
def window_task(keys):
    from collections import Counter
    return (max(Counter(keys).values()), len(keys))
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count in a dictionary. That costs O(n) time and O(k) space."

"I can do better: `Counter`. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/window-with-no-partition)

<a id="q19"></a>
### Q19. Salted Join

🔴 Stretch · *Fix a hot key*

Salt the big side's keys with i % s, and replicate each small-side row s times with each salt. Return (joined rows sorted, largest group size before and after, for a key appearing n times).

**Examples**

- `big = [('us', 1), ('us', 2), ('us', 3), ('in', 4)], small = {'us': 'A', 'in': 'B'}, s = 2` → `([['in', 4, 'B'], ['us', 1, 'A'], ['us', 2, 'A'], ['us', 3, 'A']], 3, 2)`

In [ ]:
def salted_join(big, small, s):
    """
    Salt the big side's keys with i % s, and replicate each small-side row s
    times with each salt. Return (joined rows sorted, largest group size before
    and after, for a key appearing n times).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(salted_join, [
    (([('us', 1), ('us', 2), ('us', 3), ('in', 4)], {'us': 'A', 'in': 'B'}, 2), ([['in', 4, 'B'], ['us', 1, 'A'], ['us', 2, 'A'], ['us', 3, 'A']], 3, 2)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build salted keys, and loop over both sides.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary of salted small rows.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Build salted keys, and loop over both sides.

```python
def salted_join(big, small, s):
    salted_small = []
    for k, v in small.items():
        for i in range(s):
            salted_small.append((f"{k}#{i}", v))
    groups, joined = {}, []
    for i, (k, v) in enumerate(big):
        key = f"{k}#{i % s}"
        groups[key] = groups.get(key, 0) + 1
        for sk, sv in salted_small:
            if sk == key:
                joined.append([k, v, sv])
    before = {}
    for k, _ in big:
        before[k] = before.get(k, 0) + 1
    return (sorted(joined), max(before.values()), max(groups.values()))
```

**Cost:** O(b·s·m) time, O(b) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary of salted small rows.

```python
def salted_join(big, small, s):
    from collections import Counter
    table = {f"{k}#{i}": v for k, v in small.items() for i in range(s)}
    keys = [f"{k}#{i % s}" for i, (k, _) in enumerate(big)]
    joined = [[k, v, table[sk]] for (k, v), sk in zip(big, keys)]
    return (sorted(joined), max(Counter(k for k, _ in big).values()), max(Counter(keys).values()))
```

**Cost:** O(b + s·m) time, O(s·m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build salted keys, and loop over both sides. That costs O(b·s·m) time and O(b) space."

"I can do better: A dictionary of salted small rows. That's O(b + s·m) time and O(s·m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/salted-join)

<a id="q20"></a>
### Q20. Debug a Slow Job

🔴 Stretch · *Find the real bottleneck*

Given stage (name, seconds), return the slowest stage's name and its share of total time rounded to 1 decimal, and 'Amdahl limit' speedup if only that stage gets infinitely faster, rounded to 2 decimals.

**Examples**

- `stages = [('read', 10), ('shuffle', 80), ('write', 10)]` → `('shuffle', 80.0, 5.0)`

In [ ]:
def bottleneck(stages):
    """
    Given stage (name, seconds), return the slowest stage's name and its share
    of total time rounded to 1 decimal, and 'Amdahl limit' speedup if only that
    stage gets infinitely faster, rounded to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(bottleneck, [
    (([('read', 10), ('shuffle', 80), ('write', 10)],), ('shuffle', 80.0, 5.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`max` and `sum`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop.

```python
def bottleneck(stages):
    total, name, worst = 0, None, -1
    for n, s in stages:
        total += s
        if s > worst:
            name, worst = n, s
    return (name, round(100 * worst / total, 1), round(total / (total - worst), 2))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `max` and `sum`.

```python
def bottleneck(stages):
    name, worst = max(stages, key=lambda x: x[1])
    total = sum(s for _, s in stages)
    return (name, round(100 * worst / total, 1), round(total / (total - worst), 2))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(n) time and O(1) space."

"I can do better: `max` and `sum`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/spark-internals/debug-a-slow-job)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What is a broadcast join, and when should you use it?</b></summary>

The small table is sent to all executors, so no shuffle's needed. Use it when one side fits comfortably in memory.

</details>

<details>
<summary><b>2. What is data skew, and how do you fix it?</b></summary>

Uneven key sizes make a few tasks huge. Filter NULLs, enable AQE, broadcast, isolate hot keys or salt.

</details>

<details>
<summary><b>3. What does AQE do?</b></summary>

It re-optimises during execution using real sizes: merging small partitions, splitting skewed ones and switching join strategies.

</details>

<details>
<summary><b>4. How do you decide shuffle partitions?</b></summary>

Aim for about 100–200 MB per partition, based on shuffle size. With AQE, set it high and let it coalesce.

</details>

<details>
<summary><b>5. Sort-merge vs broadcast hash join?</b></summary>

Sort-merge handles any size by shuffling and sorting both sides. Broadcast avoids shuffles, but one side must be small.

</details>

<details>
<summary><b>6. Driver OOM vs executor OOM?</b></summary>

Driver OOM usually comes from bringing data back, as with collect or a large broadcast. Executor OOM comes from skew, big partitions or too little memory.

</details>

<details>
<summary><b>7. Scenario: a daily job suddenly takes 10× longer with no code change. What do you check?</b></summary>

Data volume, new skew from a key, small files upstream, schema or plan changes, and cluster contention. Compare with the last good run in the Spark UI.

</details>

<details>
<summary><b>8. Scenario: a job writes 50,000 tiny files. How do you fix it?</b></summary>

Repartition by the partition column before writing, tune the shuffle partitions, set maxRecordsPerFile, or run compaction afterward.

</details>

<details>
<summary><b>9. Scenario: your join output is much larger than either input. What's happening?</b></summary>

A many-to-many key match is exploding rows. Check key uniqueness on both sides, and aggregate or deduplicate before joining.

</details>

<details>
<summary><b>10. Scenario: you keep adding memory but it still fails. What now?</b></summary>

Stop adding it, and find the cause: one skewed task, a collect, or a huge cartesian join. Memory rarely fixes skew.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Join strategies: the biggest performance lever
- [ ] I can explain: Skew: why jobs get stuck at 99%
- [ ] I can explain: Shuffle tuning and adaptive execution
- [ ] I can explain: Memory and out-of-memory errors
- [ ] I can explain: A debugging routine for any slow job

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Dynamic Partition Pruning
- [ ] [Q17](#q17) Sort-Merge Spill
- [ ] [Q18](#q18) Window With No Partition
- [ ] [Q19](#q19) Salted Join
- [ ] [Q20](#q20) Debug a Slow Job

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.